# AutoPilot — Ultimate Dual-GPU Pipeline
Run the **entire** AutoPilot pipeline purely inside Kaggle!

- Uses **both T4 GPUs** simultaneously (2x generation speed)
- No local execution required
- Outputs are saved straight to Kaggle's `/output` directory

In [ ]:
# ── Cell 1: Environment Setup & Clone Repo ────────────────────────────────
import os, sys, subprocess

CLONE_DIR = '/kaggle/working/autopilot'

print('[1/3] Pinning numpy...\n')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy==1.26.4'])

print('[2/3] Cloning AutoPilot repo...\n')
if not os.path.exists(CLONE_DIR):
    subprocess.run(['git', 'clone', 'https://github.com/rajatsarswat2001/autopilot.git', CLONE_DIR])

print('[3/3] Installing Kaggle requirements (this takes a few minutes)...\n')
subprocess.run([sys.executable, os.path.join(CLONE_DIR, 'kaggle_setup.py')])

print('\n✅ Setup complete! Note: if this is your first run, RESTART KERNEL now to load new packages.\n')

In [ ]:
# ── Cell 2: Download Wan 2.2 Models & ComfyUI ────────────────────────────
import os

COMFY_DIR = '/kaggle/working/ComfyUI'
if not os.path.exists(COMFY_DIR):
    os.system('git clone https://github.com/comfyanonymous/ComfyUI.git /kaggle/working/ComfyUI')

BASE = 'https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files'
WAN21_BASE = 'https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files'

DIRS = {
    'diffusion': f'{COMFY_DIR}/models/diffusion_models',
    'text_enc' : f'{COMFY_DIR}/models/text_encoders',
    'vae'      : f'{COMFY_DIR}/models/vae',
    'clip_vis' : f'{COMFY_DIR}/models/clip_vision',
}
for d in DIRS.values(): os.makedirs(d, exist_ok=True)

def dl(url, dest_dir, fname):
    dest = os.path.join(dest_dir, fname)
    if not os.path.exists(dest):
        print(f'📥 Downloading {fname} ...')
        os.system(f"aria2c --console-log-level=error -c -x 16 -s 16 -k 1M '{url}' -d '{dest_dir}' -o '{fname}'")

dl(f'{BASE}/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors', DIRS['diffusion'], 'wan2.2_ti2v_5B_fp16.safetensors')
dl(f'{BASE}/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors', DIRS['text_enc'], 'umt5_xxl_fp8_e4m3fn_scaled.safetensors')
dl(f'{BASE}/vae/wan2.2_vae.safetensors', DIRS['vae'], 'wan2.2_vae.safetensors')
dl(f'{WAN21_BASE}/clip_vision/clip_vision_h.safetensors', DIRS['clip_vis'], 'clip_vision_h.safetensors')

print('✅ Models ready!')

In [ ]:
# ── Cell 3: Configure Pipeline Environment ────────────────────────────────
import os
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
except ImportError:
    secrets = None

KEYS = ['NVIDIA_API_KEY', 'PEXELS_API_KEYS', 'GROQ_API_KEYS', 'OPENAI_API_KEY']
env_content = []

if secrets:
    for k in KEYS:
        try:
            val = secrets.get_secret(k)
            if val:
                env_content.append(f'{k}={val}')
        except:
            pass

# CRITICAL: Point AutoPilot to our internal Dual-GPU Load Balancer
env_content.extend([
    'KAGGLE_NGROK_URL=http://127.0.0.1:8080',
    'VISUAL_PARALLEL_WORKERS=2',
    'VIDEO_GEN_MODEL=wan22',
    'VIDEO_GEN_ENABLED=1'
])

with open('/kaggle/working/autopilot/autopilot_pipeline/.env', 'w') as f:
    f.write('\n'.join(env_content))

print('✅ .env written securely')

In [ ]:
# ── Cell 4: Launch Dual-GPU ComfyUI Workers + Load Balancer ──────────────
import os, time, subprocess, sys

worker_code = """
import gc, os, random, sys, torch, imageio, numpy as np
from pathlib import Path
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.responses import FileResponse
import uvicorn

sys.path.insert(0, '/kaggle/working/ComfyUI')
from nodes import NODE_CLASS_MAPPINGS
OUTPUT_DIR = '/kaggle/working/output'
INPUT_DIR  = '/kaggle/working/input'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(INPUT_DIR, exist_ok=True)

app = FastAPI()
@app.get('/health')
def health(): return {'status': 'ok'}

@app.post('/generate_video')
async def api_generate(
    image: UploadFile = File(None), prompt: str = Form(...),
    seed: int = Form(0), steps: int = Form(30),
    width: int = Form(832), height: int = Form(480), frames: int = Form(49)
):
    # Load model and generate video
    if seed == 0: seed = random.randint(0, 2**32-1)
    with torch.inference_mode():
        clip = NODE_CLASS_MAPPINGS['CLIPLoader']().load_clip('umt5_xxl_fp8_e4m3fn_scaled.safetensors', 'wan', 'default')[0]
        pos_cond = NODE_CLASS_MAPPINGS['CLIPTextEncode']().encode(clip, prompt)[0]
        neg_cond = NODE_CLASS_MAPPINGS['CLIPTextEncode']().encode(clip, 'text, watermark')[0]
        del clip; gc.collect(); torch.cuda.empty_cache()

        loaded_image, clip_vis_out = None, None
        if image is not None and image.filename:
            img_path = f'{INPUT_DIR}/{image.filename}'
            with open(img_path, 'wb') as f: f.write(await image.read())
            loaded_image = NODE_CLASS_MAPPINGS['LoadImage']().load_image(img_path)[0]
            clip_vis = NODE_CLASS_MAPPINGS['CLIPVisionLoader']().load_clip('clip_vision_h.safetensors')[0]
            clip_vis_out = NODE_CLASS_MAPPINGS['CLIPVisionEncode']().encode(clip_vis, loaded_image, 'none')[0]
            del clip_vis; gc.collect(); torch.cuda.empty_cache()

        vae = NODE_CLASS_MAPPINGS['VAELoader']().load_vae('wan2.2_vae.safetensors')[0]
        wan_cls = NODE_CLASS_MAPPINGS.get('WanImageToVideo') or NODE_CLASS_MAPPINGS.get('Wan TI2V Encode')
        if loaded_image is not None and wan_cls:
            pos_cond, neg_cond, lat = wan_cls().encode(pos_cond, neg_cond, vae, width, height, frames, 1, loaded_image, clip_vis_out)
        else:
            lat = NODE_CLASS_MAPPINGS['EmptyLatentImage']().generate(width, height, 1)[0]

        model = NODE_CLASS_MAPPINGS['UNETLoader']().load_unet('wan2.2_ti2v_5B_fp16.safetensors', 'default')[0]
        sampled = NODE_CLASS_MAPPINGS['KSampler']().sample(model, seed, steps, 6.0, 'euler', 'simple', pos_cond, neg_cond, lat, 1.0)[0]
        del model; gc.collect(); torch.cuda.empty_cache()

        decoded = NODE_CLASS_MAPPINGS['VAEDecode']().decode(vae, sampled)[0]
        del vae, sampled; gc.collect(); torch.cuda.empty_cache()

        out_path = f'{OUTPUT_DIR}/wan22_{seed}.mp4'
        frames_np = [(f.cpu().numpy() * 255).astype(np.uint8) for f in decoded]
        with imageio.get_writer(out_path, fps=16) as writer:
            for frame in frames_np: writer.append_data(frame)

        return FileResponse(out_path, media_type='video/mp4', filename='output.mp4')
if __name__ == '__main__':
    uvicorn.run(app, host='127.0.0.1', port=int(sys.argv[1]))
"""
with open('worker.py', 'w') as f: f.write(worker_code)

balancer_code = """
import httpx, uvicorn, asyncio, gc
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.responses import FileResponse

app = FastAPI()
WORKERS = ['http://127.0.0.1:8001', 'http://127.0.0.1:8002']
idx = 0; lock = asyncio.Lock()

@app.get('/health')
def health(): return {'status': 'ok'}

@app.post('/generate_video')
async def gen(image: UploadFile = File(None), prompt: str = Form(...), seed: int = Form(0), steps: int = Form(30), width: int = Form(832), height: int = Form(480), frames: int = Form(49)):
    global idx
    async with lock:
        target = WORKERS[idx]
        idx = (idx + 1) % len(WORKERS)
    data = {'prompt': prompt, 'seed': str(seed), 'steps': str(steps), 'width': str(width), 'height': str(height), 'frames': str(frames)}
    files = {}
    if image and image.filename:
        files = {'image': (image.filename, await image.read(), image.content_type)}
    async with httpx.AsyncClient(timeout=3600.0) as client:
        resp = await client.post(f'{target}/generate_video', data=data, files=files)
        out_file = f'/kaggle/working/output/proxied_{seed}.mp4'
        with open(out_file, 'wb') as f: f.write(resp.content)
        gc.collect()
        return FileResponse(out_file, media_type='video/mp4')
if __name__ == '__main__':
    uvicorn.run(app, host='0.0.0.0', port=8080)
"""
with open('balancer.py', 'w') as f: f.write(balancer_code)

print('Starting Load Balancer (Port 8080)...')
subprocess.Popen([sys.executable, 'balancer.py'])

print('Starting Worker 0 on GPU 0 (Port 8001)...')
env0 = os.environ.copy()
env0['CUDA_VISIBLE_DEVICES'] = '0'
subprocess.Popen([sys.executable, 'worker.py', '8001'], env=env0)

print('Starting Worker 1 on GPU 1 (Port 8002)...')
env1 = os.environ.copy()
env1['CUDA_VISIBLE_DEVICES'] = '1'
subprocess.Popen([sys.executable, 'worker.py', '8002'], env=env1)

import requests
for p in [8080, 8001, 8002]:
    while True:
        try:
            requests.get(f'http://127.0.0.1:{p}/health')
            print(f'✅ Port {p} is UP')
            break
        except:
            time.sleep(2)


In [ ]:
# ── Cell 5: Run AutoPilot Pipeline! ──────────────────────────────────────
import os, sys, subprocess

PIPELINE_DIR = '/kaggle/working/autopilot/autopilot_pipeline'
NICHE = 'personal_finance'
TOPIC = ''

print(f'\n🚀 Starting Pipeline (Niche: {NICHE})')
cmd = [sys.executable, 'main.py', '--niche', NICHE, '--no-db', '--approve', '--log-format', 'console']
if TOPIC: cmd.extend(['--topic', TOPIC])

proc = subprocess.Popen(cmd, cwd=PIPELINE_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout: print(line, end='', flush=True)
proc.wait()

print(f'\n✅ Pipeline Finished with code {proc.returncode}\nCheck /kaggle/working/autopilot/autopilot_pipeline/outputs/video for MP4 files')